In [13]:
import numpy as np
np.random.seed(42)

In [14]:
import sys
sys.path.append("../src")

import importlib
import utils
importlib.reload(utils)

from utils import add_enrollment_per_1000, clean_analysis_df

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

figures_dir = Path("../figures")
figures_dir.mkdir(exist_ok = True)

#load mergered dataset
merged_df = pd.read_csv("../data/processed/MD_merged_crime_college.csv")

In [16]:
required = ["crime_rate", "college_enrollment_total", "population"]
missing = [c for c in required if c not in merged_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Available: {list(merged_df.columns)}")

merged_df = add_enrollment_per_1000(merged_df)
merged_df = clean_analysis_df(merged_df)

df = merged_df[["crime_rate", "enrollment_per_1000", "population"]].dropna().copy()

print(df.columns.tolist())
df.shape


['crime_rate', 'enrollment_per_1000', 'population']


(24, 3)

H0: Neighborhood violent crime rate has no association with college enrollment per capita.

H1: Higher violent crime rate is associated with lower college enrollment per capita.

In [17]:
#correlation tests with p-values
x = df["crime_rate"].to_numpy()
y = df["enrollment_per_1000"].to_numpy()

pearson_r, pearson_p = stats.pearsonr(x, y)
spearman_rho, spearman_p = stats.spearmanr(x, y)

print(f"Pearson r = {pearson_r:.3f}, p = {pearson_p:.4g}")
print(f"Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.4g}")

Pearson r = 0.373, p = 0.07233
Spearman rho = 0.237, p = 0.2655


Pearson - linear relationship, it's sensitive to outliers
Spearman - monotonic relationship, it's more robust to skew/outliers
A small p-value means the observed association is unlikely under H0

In [18]:
#regression coefficienet test
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
print(f"Slope (crime_rate coefficient): {slope:.6f}")
print(f"Intercept: {intercept:.3f}")
print(f"R-squared: {r_value**2:.3f}")
print(f"P-value for slope: {p_value:.4g}")
print(f"Standard error: {std_err:.6f}")

#save regression diagnostic plot
plt.figure()
plt.scatter(x, y, alpha=0.6)

x_line = np.linspace(x.min(), x.max(), 100)
y_line = intercept + slope * x_line

plt.plot(x_line, y_line)

plt.title("Enrollment per 1,000 vs Crime Rate")
plt.xlabel("Crime rate (per 100,000)")
plt.ylabel("Enrollment per 1,000")

plt.tight_layout()
plt.savefig("../figures/OLS_fit_enrollment_vs_crime.png", dpi=200)
plt.close()


Slope (crime_rate coefficient): 0.045111
Intercept: 28.286
R-squared: 0.139
P-value for slope: 0.07233
Standard error: 0.023897


In [ ]:
#check for robustness
low, high = df["crime_rate"].quantile([0.01, 0.99])
df_trim = df[(df["crime_rate"] >= low) & (df["crime_rate"] <= high)].copy()

x_t = df_trim["crime_rate"].to_numpy()
y_t = df_trim["enrollment_per_1000"].to_numpy()

pearson_r_t, pearson_p_t = stats.pearsonr(x_t, y_t)
spearman_rho_t, spearman_p_t = stats.spearmanr(x_t, y_t)

print("Trimmed (1st - 99th percentile of crime_rate):")
print(f"Pearson r = {pearson_r_t:.3f}, p = {pearson_p_t:.4g}")
print(f"Spearman rho = {spearman_rho_t:.3f}, p = {spearman_p_t:.4g}")

Trimmed (1st - 99th percentile of crime_rate):
Pearson r = 0.422, p = 0.05023
Spearman rho = 0.129, p = 0.5675


In [ ]:
#save statistical results
results = pd.DataFrame([
    {"test": "pearson", "stat": pearson_r, "p-value": pearson_p, "n": len(df)},
    {"test": "spearman", "stat": spearman_rho, "p-value": spearman_p, "n":len(df)},
    {"test": "pearson_trim", "stat": pearson_r_t, "p-value": pearson_p_t, "n": len(df_trim)},
    {"test": "spearman_trim", "stat": spearman_rho_t, "p-value": spearman_p_t, "n": len(df_trim)},
])

results
results.to_csv("../data/processed/statistical_results.csv", index=False)

We conducted Pearson and Spearman correlation tests and an OLS regression coefficient test using SciPy. Results indicate a moderate positive association between violent crime rates and college enrollment per capita. The Pearson correlation was r = 0.373 (p = 0.072, n = 24), while the Spearman correlation was weaker (ρ = 0.237, p = 0.265, n = 24), suggesting sensitivity to distributional skew and outliers.

After excluding extreme crime-rate counties (1st–99th percentile trimming), the Pearson correlation increased to r = 0.422 (p = 0.050, n = 22), while the Spearman correlation remained weak (ρ = 0.129, p = 0.568, n = 22). This indicates that the observed relationship is driven primarily by linear effects and is moderately influenced by extreme observations.

The regression coefficient test showed a statistically marginal result consistent with the Pearson correlation, with an estimated effect size of approximately R² ≈ 0.14–0.18, indicating that violent crime alone explains a limited proportion of the variance in college enrollment per capita.

From these results and statistical knowledge, aossociation does not equal cauasation. We have missing confounding variables such as income, unemployment, schoo funding, etc. It also had ecological fallacy risk, for example, county-level aggregation.